# Student Performance Factors — EDA

**Course:** Introduction to Data Science  
**Topic:** Exploratory Data Analysis (EDA)

## Objective
This notebook explores how study habits and other student-related factors are associated with exam scores. It follows the EDA steps discussed in Lecture 06.

The graphs describe patterns in this dataset. They do not prove that one factor causes a higher or lower score.

## 1. Import libraries

We will use Pandas to work with the data, and Matplotlib and Seaborn to make the graphs.

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load the dataset

Before running this notebook, add **Student Performance Factors** as an input dataset in Kaggle. The code looks for the CSV file inside Kaggle's input folder.

In [ ]:
csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

matching_files = [
    file for file in csv_files
    if os.path.basename(file).lower() == "studentperformancefactors.csv"
]

if not matching_files:
    print("CSV files available in Kaggle input:")
    for file in csv_files:
        print(file)
    raise FileNotFoundError(
        "Add the Student Performance Factors dataset in Kaggle, "
        "then check that StudentPerformanceFactors.csv is available."
    )

df = pd.read_csv(matching_files[0])

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)
df.head()

## 3. Understand and clean the data

First, check the column names, data types, missing values, and duplicate rows. Missing numerical values are filled with the median, and missing category values are filled with the most common category.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-missing values:")
df.info()

print("\\nMissing values in each column:")
print(df.isnull().sum())

print("\\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().copy()

numeric_cols = df.select_dtypes(include="number").columns
categorical_cols = df.select_dtypes(exclude="number").columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Shape after cleaning:", df.shape)
print("Missing values left:", df.isnull().sum().sum())

## 4. Basic summary

The summary helps us see the typical values and the spread of the numerical columns.

In [ ]:
df.describe().round(2)

## 5. Histogram — exam score distribution

A histogram shows how exam scores are spread across score ranges. Look for the ranges where scores appear most often.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="Exam_Score", bins=20, kde=False)
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam score")
plt.ylabel("Number of students")
plt.tight_layout()
plt.savefig("/kaggle/working/01_histogram_exam_score.png", dpi=150)
plt.show()

**What to notice:** The tallest bars show the score ranges that occur most often. The graph also helps us see whether scores are spread out or concentrated in a smaller range.

## 6. Density plot — exam score distribution

A density plot shows the general shape of the score distribution using a smooth curve.

In [ ]:
plt.figure(figsize=(7, 4))
sns.kdeplot(data=df, x="Exam_Score", fill=True)
plt.title("Density Plot of Exam Scores")
plt.xlabel("Exam score")
plt.ylabel("Density")
plt.tight_layout()
plt.savefig("/kaggle/working/02_density_exam_score.png", dpi=150)
plt.show()

**What to notice:** Peaks show where scores are more concentrated. The shape should be interpreted from the graph rather than assumed in advance.

## 7. Bar chart — student motivation levels

A bar chart compares the number of students in each motivation category.

In [ ]:
if "Motivation_Level" in df.columns:
    plt.figure(figsize=(7, 4))
    order = df["Motivation_Level"].value_counts().index
    sns.countplot(data=df, x="Motivation_Level", order=order)
    plt.title("Number of Students by Motivation Level")
    plt.xlabel("Motivation level")
    plt.ylabel("Number of students")
    plt.tight_layout()
    plt.savefig("/kaggle/working/03_bar_motivation_level.png", dpi=150)
    plt.show()
else:
    print("Motivation_Level was not found in this dataset.")

**What to notice:** Taller bars represent categories with more students. This compares category sizes; it does not show which category has higher exam scores.

## 8. Boxplot — exam scores

A boxplot shows the median, the middle 50% of scores, the overall spread, and possible outliers.

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="Exam_Score")
plt.title("Spread of Exam Scores")
plt.xlabel("Exam score")
plt.tight_layout()
plt.savefig("/kaggle/working/04_boxplot_exam_score.png", dpi=150)
plt.show()

**What to notice:** The line inside the box is the median. Points beyond the whiskers may be unusual values, but they should be checked before deciding whether they are errors.

## 9. Feature vs. target — attendance groups and exam scores

Here, attendance is grouped into ranges so we can compare exam scores across groups. This is a simple way to examine a possible relationship with the target column, `Exam_Score`.

In [ ]:
attendance_bins = [0, 50, 60, 70, 80, 90, 100]
attendance_labels = ["0–50", "51–60", "61–70", "71–80", "81–90", "91–100"]

df["Attendance_Group"] = pd.cut(
    df["Attendance"],
    bins=attendance_bins,
    labels=attendance_labels,
    include_lowest=True
)

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Attendance_Group", y="Exam_Score")
plt.title("Exam Scores Across Attendance Groups")
plt.xlabel("Attendance (%)")
plt.ylabel("Exam score")
plt.tight_layout()
plt.savefig("/kaggle/working/05_attendance_vs_exam_score.png", dpi=150)
plt.show()

**What to notice:** Compare the medians and spreads across attendance groups. The graph may show an association, but it cannot prove that attendance alone caused a score difference.

## 10. Scatter plot — study hours and exam scores

Each dot represents a record. The scatter plot helps us see whether study hours and exam scores tend to move together.

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(
    data=df,
    x="Hours_Studied",
    y="Exam_Score",
    alpha=0.5
)
plt.title("Study Hours and Exam Scores")
plt.xlabel("Hours studied per week")
plt.ylabel("Exam score")
plt.tight_layout()
plt.savefig("/kaggle/working/06_scatter_study_hours_exam_score.png", dpi=150)
plt.show()

**What to notice:** Look for an upward, downward, or unclear pattern. Individual points can vary, so study hours should not be treated as the only factor related to exam scores.

## 11. Line plot — average score by study-hour group

The dataset does not provide a time sequence, so this line plot compares average scores across ordered study-hour groups. It is not a trend over time.

In [ ]:
df["Study_Hour_Group"] = pd.cut(
    df["Hours_Studied"],
    bins=5,
    include_lowest=True
)

average_scores = (
    df.groupby("Study_Hour_Group", observed=False)["Exam_Score"]
    .mean()
    .dropna()
)

plt.figure(figsize=(8, 4))
plt.plot(range(len(average_scores)), average_scores.values, marker="o")
plt.xticks(
    range(len(average_scores)),
    [str(group) for group in average_scores.index],
    rotation=25,
    ha="right"
)
plt.title("Average Exam Score Across Study-Hour Groups")
plt.xlabel("Study-hour group")
plt.ylabel("Average exam score")
plt.tight_layout()
plt.savefig("/kaggle/working/07_line_study_groups.png", dpi=150)
plt.show()

**What to notice:** Compare the average scores between groups and see whether they generally rise, fall, or stay similar. These are grouped averages, not individual predictions.

## 12. Heatmap — correlations (last graph)

The heatmap is placed at the end of the visualization section. It shows correlations between numerical columns. Values closer to +1 indicate a positive linear association, values closer to -1 indicate a negative linear association, and values near 0 indicate a weak linear association.

In [ ]:
numeric_data = df.select_dtypes(include="number")
correlation = numeric_data.corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.savefig("/kaggle/working/08_correlation_heatmap.png", dpi=150)
plt.show()

**What to notice:** Check which numerical features have the strongest positive or negative correlations with `Exam_Score`. Correlation does not prove cause and effect.

## 13. Short conclusion

After running the notebook, write a few sentences here using the patterns you actually see in your graphs. Mention one pattern in exam scores, one relationship with study hours or attendance, and one point from the heatmap. Avoid claiming that a relationship proves cause and effect.

## 14. Save the graphs

The graph images are saved in `/kaggle/working/`. After running the notebook, use Kaggle's output/download options if you need to keep the images for your GitHub repository.